# 🩺 Analítica Predictiva de Complicaciones en Dengue Pediátrico
**Estudio Clínico:** *F1000Research (2022)* - *"Predictive scoring system for risk of complications in pediatric dengue infection"*  
**Autores:** Dr. Monisha Bhaskar, Dr. Soundarya Mahalingam, Dr. Harish M M, Dr. Basavaprabhu Achappa (Kasturba Medical College, Mangalore / Manipal Academy of Higher Education, India).  
**DOI:** [10.12688/f1000research.111214.1](https://doi.org/10.12688/f1000research.111214.1)  
**Población:** 303 pacientes pediátricos (0 a 18 años) confirmados serológicamente.

---
## 🧠 Enfoque de Negocio y Dimensión Humana (Saber y Ser)
### Consecuencias Operativas de un Mal Preprocesamiento en Triaje Pediátrico:
El dengue pediátrico presenta una ventana crítica tras la caída de la fiebre (días 3 a 7) donde el paciente puede evolucionar súbitamente a choque hipovolémico por fuga capilar (Dengue Shock Syndrome - DSS), hemorragia masiva o falla multiorgánica.

1. **Peligro Crítico del Falso Negativo (FN):**
   - *Causa:* Imputación errónea de trombocitopenia o hematocrito, o **invertir la codificación de la presión arterial al ingreso** (`BP AT ADMISSION`, donde 1=Normotensión y 2=Hipotensión).
   - *Consecuencia:* Un niño en fase crítica de fuga plasmática es clasificado erróneamente como "Bajo Riesgo" y dado de alta a su hogar. Horas después colapsa hemodinámicamente en su domicilio o llega a urgencias en paro cardiorrespiratorio prevenible (**mortalidad pediátrica evitable**).

2. **Impacto Operativo del Falso Positivo (FP):**
   - *Causa:* Umbrales no calibrados pediátricamente o normalizaciones defectuosas.
   - *Consecuencia:* Hospitalización innecesaria de pacientes con dengue clásico benigno en salas de aislamiento o camas de Cuidados Intensivos Pediátricos (UCIP). Saturación de camas en epidemias, sobrecostos hospitalarios ($800 a $3,500 USD por paciente) y riesgo iatrogénico de sobrehidratación (edema agudo de pulmón por exceso de líquidos intravenosos).
---

## 🧹 CELDA 1: Carga, Limpieza, Tratamiento de Nulos y Feature Engineering

In [1]:
import os
import numpy as np
import pandas as pd

# 1. Cargar el dataset original de dengue pediátrico
posibles_rutas = [
    'data/MASTER_CHART_F1000Research.xlsx',
    'MASTER_CHART_F1000Research (1).xlsx',
    'MASTER_CHART_F1000Research.xlsx'
]
ruta_excel = next((r for r in posibles_rutas if os.path.exists(r)), None)
if not ruta_excel:
    raise FileNotFoundError('No se encontro el archivo Excel del dataset.')

df = pd.read_excel(ruta_excel)
df.columns = df.columns.str.strip()
print(f'Dataset cargado exitosamente: {df.shape[0]} filas, {df.shape[1]} columnas.\n')

# 2. Análisis y tratamiento riguroso de valores nulos
# Fila 283 presenta nulos en ALT y CREATININE. Se imputan con la mediana clinica del grupo no complicado:
mediana_alt = df[df['GROUPS: DENGUE FEVER OR COMPLICATED DENGUE'] == 1]['ALT'].median()
mediana_creat = df[df['GROUPS: DENGUE FEVER OR COMPLICATED DENGUE'] == 1]['CREATININE'].median()
df['ALT'] = df['ALT'].fillna(mediana_alt)
df['CREATININE'] = df['CREATININE'].fillna(mediana_creat)
print(f'Imputacion de nulos completada: ALT = {mediana_alt} U/L, CREATININE = {mediana_creat} mg/dL.')

# 3. Eliminacion de columnas constantes, redundantes o con fuga de datos (Data Leakage)
# - FERRITIN: 112 valores nulos (37%), excluida por falta de disponibilidad en triaje de emergencia.
# - IV FLUIDS, ANTIBIOTICS, ANTIPYRETICS, STEROIDS: Fuga de datos (tratamientos post-ingreso).
# - BREATHLESSNESS: Solo 2 casos positivos en todo el estudio (p=0.307, varianza nula).
# - LOWEST WBC: Colineal con HIGHEST WBC (r > 0.88).
cols_descartadas = ['FERRITIN', 'IV FLUIDS', 'ANTIBIOTICS', 'ANTIPYRETICS', 'STEROIDS', 'BREATHLESSNESS', 'LOWEST WBC']
df_limpio = df.drop(columns=cols_descartadas)
print(f'Columnas descartadas justificadamente: {cols_descartadas}\n')

# 4. Conversion de variables categoricas/texto a numericas binarias (0 y 1)
# Target: 1 (No complicado) -> 0, 2 (Complicado) -> 1
df_limpio['TARGET_COMPLICADO'] = (df_limpio['GROUPS: DENGUE FEVER OR COMPLICATED DENGUE'] == 2).astype(int)
df_limpio['EDAD_MAYOR_6'] = (df_limpio['AGE'] == 2).astype(int)
df_limpio['GENERO_FEMENINO'] = (df_limpio['GENDER'] == 2).astype(int)

# Sintomas: 1=Si (Presente) -> 1, 2=No (Ausente) -> 0
sintomas = ['HEADACHE', 'MYALGIA', 'ABDOMINAL PAIN', 'RASH', 'VOMITING', 'BLEEDING', 'ORGANOMEGALY']
for s in sintomas:
    df_limpio[s] = (df_limpio[s] == 1).astype(int)

# ATENCION CRITICA: BP AT ADMISSION en dataset original: 1=Normotension, 2=Hipotension
# Lo recodificamos como HIPOTENSION (1=Hipotension presente, 0=Normal):
df_limpio['HIPOTENSION'] = (df_limpio['BP AT ADMISSION'] == 2).astype(int)

# 5. Feature Engineering Medico (Umbrales ROC y Score Clinico F1000Research)
df_limpio['PCV_GE_42'] = (df_limpio['PCV'] >= 42.0).astype(int)               # Hemoconcentracion / fuga capilar
df_limpio['PLAQUETAS_LE_75K'] = (df_limpio['PLATELET COUNT'] <= 75000).astype(int) # Trombocitopenia de alto riesgo
df_limpio['WBC_GE_7K'] = (df_limpio['HIGHEST WBC'] >= 7000).astype(int)       # Respuesta inflamatoria sistemica
df_limpio['ALT_GE_70'] = (df_limpio['ALT'] >= 70.0).astype(int)               # Transaminitis hepatica
df_limpio['FIEBRE_GE_6D'] = (df_limpio['DURATION OF FEVER'] >= 6).astype(int) # Fiebre prolongada

# Score Clinico Ponderado (Tabla 3 F1000Research: Rango 0-7 puntos):
df_limpio['SCORE_CLINICO'] = (
    df_limpio['HIPOTENSION'] * 3 +
    df_limpio['PCV_GE_42'] * 1 +
    df_limpio['PLAQUETAS_LE_75K'] * 1 +
    df_limpio['WBC_GE_7K'] * 1 +
    df_limpio['ALT_GE_70'] * 1
)
# Bioindice de fuga capilar / trombocitopenia:
df_limpio['INDICE_HEMO_PLAQUETA'] = (df_limpio['PCV'] / (df_limpio['PLATELET COUNT'] + 1e-5)) * 100000.0
df_limpio['HB_PCV_RATIO'] = df_limpio['HB'] / (df_limpio['PCV'] + 1e-5)

print('Distribucion de la variable objetivo (TARGET_COMPLICADO):')
print(df_limpio['TARGET_COMPLICADO'].value_counts().rename({0: 'No Complicado (0)', 1: 'Complicado (1)'}))
print('\nDistribucion del Score Clinico (0 a 7 puntos):')
print(df_limpio['SCORE_CLINICO'].value_counts().sort_index())


Dataset cargado exitosamente: 303 filas, 25 columnas.

Imputacion de nulos completada: ALT = 25.0 U/L, CREATININE = 0.5 mg/dL.
Columnas descartadas justificadamente: ['FERRITIN', 'IV FLUIDS', 'ANTIBIOTICS', 'ANTIPYRETICS', 'STEROIDS', 'BREATHLESSNESS', 'LOWEST WBC']

Distribucion de la variable objetivo (TARGET_COMPLICADO):
TARGET_COMPLICADO
No Complicado (0)    240
Complicado (1)        63
Name: count, dtype: int64

Distribucion del Score Clinico (0 a 7 puntos):
SCORE_CLINICO
0    93
1    91
2    59
3    26
4     9
5    15
6     5
7     5
Name: count, dtype: int64


## 🤖 CELDA 2: Modelado Predictivo (Regresión Logística vs. Red Neuronal Artificial ANN)

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

# Definir vector de features de entrada
features = [
    'EDAD_MAYOR_6', 'GENERO_FEMENINO', 'DURATION OF FEVER', 'VOMITING',
    'ABDOMINAL PAIN', 'BLEEDING', 'HIPOTENSION', 'ORGANOMEGALY', 'HB',
    'PCV', 'HIGHEST WBC', 'PLATELET COUNT', 'ALT', 'PCV_GE_42',
    'PLAQUETAS_LE_75K', 'WBC_GE_7K', 'ALT_GE_70', 'FIEBRE_GE_6D',
    'SCORE_CLINICO', 'INDICE_HEMO_PLAQUETA', 'HB_PCV_RATIO'
]

X = df_limpio[features].values
y = df_limpio['TARGET_COMPLICADO'].values

# Train / Test split estratificado (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Muestras de entrenamiento: {X_train.shape[0]} | Muestras de prueba: {X_test.shape[0]} (Complicados: {sum(y_test)})')

# =====================================================================
# MODELO 1: REGRESION LOGISTICA (Scikit-Learn)
# =====================================================================
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42, C=0.5)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)
y_proba_lr = lr.predict_proba(X_test_scaled)[:, 1]

acc_lr = accuracy_score(y_test, y_pred_lr)
prec_lr = precision_score(y_test, y_pred_lr)
rec_lr = recall_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr)
auc_lr = roc_auc_score(y_test, y_proba_lr)
cm_lr = confusion_matrix(y_test, y_pred_lr)

# =====================================================================
# MODELO 2: RED NEURONAL ARTIFICIAL - ANN (TensorFlow/Keras)
# =====================================================================
tf.random.set_seed(42)
np.random.seed(42)

# Pesos de clase para balancear el 4:1
w0 = (1 / np.mean(y_train == 0)) / 2.0
w1 = (1 / np.mean(y_train == 1)) / 2.0
class_weights = {0: w0, 1: w1}

ann = keras.Sequential([
    layers.Input(shape=(len(features),)),
    layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(0.005)),
    layers.Dropout(0.30),
    layers.Dense(16, activation='relu', kernel_regularizer=regularizers.l2(0.005)),
    layers.Dropout(0.20),
    layers.Dense(8, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

ann.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.003),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)

ann.fit(
    X_train_scaled, y_train,
    epochs=120,
    batch_size=16,
    class_weight=class_weights,
    validation_split=0.15,
    verbose=0
)

y_proba_ann = ann.predict(X_test_scaled, verbose=0).ravel()
y_pred_ann = (y_proba_ann >= 0.50).astype(int)

acc_ann = accuracy_score(y_test, y_pred_ann)
prec_ann = precision_score(y_test, y_pred_ann)
rec_ann = recall_score(y_test, y_pred_ann)
f1_ann = f1_score(y_test, y_pred_ann)
auc_ann = roc_auc_score(y_test, y_proba_ann)
cm_ann = confusion_matrix(y_test, y_pred_ann)

print('=' * 70)
print(f"{'Metrica':<22} | {'Regresion Logistica':<20} | {'Red Neuronal (ANN)':<20}")
print('-' * 70)
print(f"{'Accuracy':<22} | {acc_lr:<20.4f} | {acc_ann:<20.4f}")
print(f"{'Precision':<22} | {prec_lr:<20.4f} | {prec_ann:<20.4f}")
print(f"{'Recall (Sensibilidad)':<22} | {rec_lr:<20.4f} | {rec_ann:<20.4f}")
print(f"{'F1-Score':<22} | {f1_lr:<20.4f} | {f1_ann:<20.4f}")
print(f"{'ROC-AUC':<22} | {auc_lr:<20.4f} | {auc_ann:<20.4f}")
print('=' * 70)
print(f'Matriz Confusion Regresion Logistica:\n{cm_lr}')
print(f'Matriz Confusion Red Neuronal ANN:\n{cm_ann}')


Muestras de entrenamiento: 242 | Muestras de prueba: 61 (Complicados: 13)
Metrica                | Regresion Logistica  | Red Neuronal (ANN)  
----------------------------------------------------------------------
Accuracy               | 0.8197               | 0.8197              
Precision              | 0.5455               | 0.5455              
Recall (Sensibilidad)  | 0.9231               | 0.9231              
F1-Score               | 0.6857               | 0.6857              
ROC-AUC                | 0.9712               | 0.9071              
Matriz Confusion Regresion Logistica:
[[38 10]
 [ 1 12]]
Matriz Confusion Red Neuronal ANN:
[[38 10]
 [ 1 12]]


## 💾 CELDA 3: Exportación de Modelos y Artefactos de Inferencia

In [3]:
import json
import joblib

os.makedirs('models', exist_ok=True)

# 1. Guardar Scaler para normalización en tiempo real
joblib.dump(scaler, 'models/scaler.joblib')

# 2. Guardar Modelo de Regresión Logística
joblib.dump(lr, 'models/modelo_regresion_logistica.joblib')

# 3. Guardar Modelo de Red Neuronal Artificial (ANN)
ann.save('models/modelo_ann.keras')

# 4. Guardar Configuración de Features para Inferencia en app.py
config = {
    'features': features,
    'metricas': {
        'regresion_logistica': {'accuracy': acc_lr, 'recall': rec_lr, 'roc_auc': auc_lr},
        'red_neuronal_ann': {'accuracy': acc_ann, 'recall': rec_ann, 'roc_auc': auc_ann}
    }
}
with open('models/features_config.json', 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=4)

print('Artefactos guardados exitosamente en la carpeta models/:')
for archivo in os.listdir('models'):
    print(f' - models/{archivo}')


Artefactos guardados exitosamente en la carpeta models/:
 - models/features_config.json
 - models/metricas_modelos.json
 - models/modelo_ann.keras
 - models/modelo_regresion_logistica.joblib
 - models/scaler.joblib
